[◀ Notebook 09: Classes & OOP](09_classes_and_oop.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 11: Type Hinting ▶](11_type_hinting.ipynb)

# Notebook 10: Advanced Data Model (Magic Methods)

In Python, the behavior of user-defined classes can be integrated with the language's core syntax (operators, collections, iteration) through special methods. These are referred to as **magic methods** or **dunder methods** (double underscore). 

Implementing magic methods allows you to customize how your objects interact with operators (`+`, `-`, `==`), functions (`len()`, `repr()`), and protocols (containers, iteration).

---

## 1. Object Representations: `__repr__` vs. `__str__`

Python has two built-in mechanisms to convert objects to strings:
- **`__repr__(self)`:** Intended to return an *unambiguous* string representation of the object. Ideally, the returned string should look like valid Python code that can recreate the object (e.g., `Vector(2, 3)`). Used by developers, the interactive interpreter, and fallback printing.
- **`__str__(self)`:** Intended to return a *user-friendly* string representation suitable for end-users. Used by `print()`, `str()`, and string formatting.

If a class implements `__repr__` but not `__str__`, calls to `str()` will automatically fall back to `__repr__`.

In [ ]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def __repr__(self):
        # !r specifier enforces repr representation of the inner values
        return f"Book({self.title!r}, {self.author!r})"

    def __str__(self):
        return f"'{self.title}' by {self.author}"

book = Book("1984", "George Orwell")
print("repr():", repr(book))
print("str():", str(book))
print("Direct print:", book)

## 2. Operator Overloading (Mathematics & Comparisons)

Python resolves binary operations dynamically by looking up corresponding methods on the operands:

### Mathematical Hooks
- `a + b` calls `a.__add__(b)`.
- **Reflective Methods:** If `a` does not support addition with `b` (or returns `NotImplemented`), Python looks for a reflective version on the right operand: `b.__radd__(a)`.

### Rich Comparison Hooks and `NotImplemented`
- Rich comparison methods map directly to comparison operators:
  - `a == b` calls `a.__eq__(b)`
  - `a != b` calls `a.__ne__(b)`
  - `a < b` calls `a.__lt__(b)`
  - `a <= b` calls `a.__le__(b)`
  - `a > b` calls `a.__gt__(b)`
  - `a >= b` calls `a.__ge__(b)`
- **The Role of `NotImplemented`**: If a comparison method cannot handle the type of the other operand, it should return the special singleton constant **`NotImplemented`** (which is NOT the same as raising `NotImplementedError`). Returning `NotImplemented` alerts the CPython evaluation loop to try the reflected operation on the second operand (e.g., if `a.__lt__(b)` returns `NotImplemented`, CPython will attempt `b.__gt__(a)`). If both return `NotImplemented`, CPython falls back to standard behavior (for `==` and `!=` it falls back to identity checks, and for ordering it raises `TypeError`).
- By default, custom class instances compare using identity (`is`), unless `__eq__` is overridden.

In [ ]:
class Currency:
    def __init__(self, amount, code="USD"):
        self.amount = amount
        self.code = code

    def __add__(self, other):
        if isinstance(other, Currency):
            if self.code != other.code:
                raise ValueError("Cannot add different currencies!")
            return Currency(self.amount + other.amount, self.code)
        elif isinstance(other, (int, float)):
            return Currency(self.amount + other, self.code)
        return NotImplemented

    def __radd__(self, other):
        return self.__add__(other)

    def __eq__(self, other):
        if not isinstance(other, Currency):
            return NotImplemented
        return self.amount == other.amount and self.code == other.code

    def __lt__(self, other):
        if not isinstance(other, Currency):
            return NotImplemented
        if self.code != other.code:
            raise ValueError("Cannot compare different currencies!")
        return self.amount < other.amount

    def __repr__(self):
        return f"Currency({self.amount}, {self.code!r})"

c1 = Currency(100, "USD")
c2 = Currency(50, "USD")

print("c1 + c2 =", c1 + c2)
print("c1 + 25 =", c1 + 25)
print("25 + c1 =", 25 + c1)
print("c1 == Currency(100, 'USD') =", c1 == Currency(100, "USD"))
print("c2 < c1 =", c2 < c1)


---

## 3. Dynamic Attribute Access Control

Python exposes deep hooks into the attribute lookup mechanics of classes:
- **`__getattr__(self, name)`:** Called **only** when the requested attribute is not found in the normal namespaces (i.e. not in `self.__dict__` or parent MRO). Useful for lazy loading or fallback lookups.
- **`__getattribute__(self, name)`:** Called unconditionally for **every** attribute access. Overriding this requires extreme care to avoid infinite recursion; you must route lookups via the parent implementation: `super().__getattribute__(name)`.
- **`__setattr__(self, name, value)`:** Intercepts attribute assignment.

In [ ]:
class LazyRecord:
    def __init__(self, data_dict):
        # Access dict using super to bypass customized __setattr__ if overridden
        super().__setattr__('_data', data_dict)

    def __getattr__(self, name):
        print(f"[__getattr__] '{name}' not found in normal dictionary; checking data source.")
        if name in self._data:
            return self._data[name]
        raise AttributeError(f"No attribute named {name}")

    def __setattr__(self, name, value):
        print(f"[__setattr__] Setting {name} = {value}")
        self._data[name] = value

rec = LazyRecord({"username": "alice", "role": "admin"})

# Accessing existing namespace key via __getattr__ fallback
print("username:", rec.username)

# Modifying
rec.role = "superuser"
print("role:", rec.role)

try:
    print(rec.nonexistent)
except AttributeError as e:
    print("AttributeError caught:", e)

## 4. Emulating Container Types

Your custom classes can behave like lists, dictionaries, or tuples by implementing the container protocol. This protocol allows you to emulate sequences (like list or tuple) and mappings (like dict):

### Custom Container Protocol Methods:
- **`__len__(self)`**: Invoked by `len()`. Must return an integer $\ge 0$.
- **`__getitem__(self, key)`**: Invoked by index lookup `obj[key]`.
- **`__setitem__(self, key, value)`**: Invoked by index assignment `obj[key] = value`.
- **`__delitem__(self, key)`**: Invoked by item deletion `del obj[key]`.
- **`__contains__(self, item)`**: Invoked by the `in` containment check operator.

### Handling Slices and Negative Index Normalization inside `__getitem__`
When subclassing or emulating sequences, the `key` passed to `__getitem__` may not be a simple integer:
1. **`slice` Objects**: When a user performs slicing (e.g., `obj[1:5:2]`), Python passes a `slice` object as the `key` argument. A `slice` object has `start`, `stop`, and `step` attributes. You must check `isinstance(key, slice)` and handle it accordingly. You can use the `key.indices(len(self))` helper method to obtain a tuple of `(start, stop, step)` normalized to the container's length.
2. **Negative Index Normalization**: Python supports indexing from the end using negative numbers (e.g., `obj[-1]`). To normalize a negative index, you check if `key < 0`. If so, you convert it to a positive index using `key += len(self)`. You must raise an `IndexError` if the normalized index is still out of bounds.

### Concrete Example of Custom Sequence Container:
```python
class CustomSequence:
    def __init__(self, items):
        self._data = list(items)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, key):
        if isinstance(key, slice):
            # Extract normalized indices for the slice based on current length
            start, stop, step = key.indices(len(self))
            return CustomSequence([self._data[i] for i in range(start, stop, step)])
        
        if isinstance(key, int):
            # Normalize negative index
            index = key
            if index < 0:
                index += len(self)
            if index < 0 or index >= len(self):
                raise IndexError("Index out of bounds")
            return self._data[index]
            
        raise TypeError(f"Invalid key type: {type(key)}")

    def __setitem__(self, key, value):
        if isinstance(key, int):
            index = key
            if index < 0:
                index += len(self)
            if index < 0 or index >= len(self):
                raise IndexError("Index out of bounds")
            self._data[index] = value
        else:
            raise TypeError("Index must be an integer")

    def __delitem__(self, key):
        if isinstance(key, int):
            index = key
            if index < 0:
                index += len(self)
            if index < 0 or index >= len(self):
                raise IndexError("Index out of bounds")
            del self._data[index]
        else:
            raise TypeError("Index must be an integer")
```

In [ ]:
class CustomMatrix:
    def __init__(self, rows):
        self._grid = rows

    def __len__(self):
        return len(self._grid)

    def __getitem__(self, index):
        # Custom lookup handling
        return self._grid[index]

    def __contains__(self, row):
        return row in self._grid

matrix = CustomMatrix([[1, 2], [3, 4], [5, 6]])
print("Length:", len(matrix))
print("Second Row:", matrix[1])
print("Contains [3, 4]?:", [3, 4] in matrix)

class CustomSequence:
    def __init__(self, items):
        self._data = list(items)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, key):
        # Handling slice objects
        if isinstance(key, slice):
            start, stop, step = key.indices(len(self))
            print(f"[__getitem__] Slice lookup: start={start}, stop={stop}, step={step}")
            return CustomSequence([self._data[i] for i in range(start, stop, step)])
        
        # Handling negative index normalization
        if isinstance(key, int):
            index = key
            if index < 0:
                index += len(self)
            if index < 0 or index >= len(self):
                raise IndexError("Index out of bounds")
            return self._data[index]
        
        raise TypeError(f"Invalid key type: {type(key)}")

seq = CustomSequence([10, 20, 30, 40, 50])
print("Length:", len(seq))
print("Element at negative index -1:", seq[-1])
sliced = seq[1:4:2]
print("Sliced elements:", sliced._data)


### Challenge 1: A Complete 2D Vector Class

Implement a class `Vector2D` that represents mathematical coordinates `(x, y)` supporting:
1. Initialization: `Vector2D(x, y)` converting inputs to floats.
2. Representation: `repr(v)` returns `"Vector2D(x, y)"` and `str(v)` returns `"(x, y)"`.
3. Math addition and subtraction with other `Vector2D` instances AND numeric values (applied to both coordinates). Support reflective/right-hand addition and subtraction.
4. Math multiplication with numeric scalars (`v * scalar` and `scalar * v`).
5. Division: `v / scalar` (raise `ZeroDivisionError` if scalar is zero).
6. Absolute value: `abs(v)` returns the magnitude: $\sqrt{x^2 + y^2}$.
7. Equality comparison: `v1 == v2` (returns `True` if components match exactly).

In [ ]:
import math

class Vector2D:
    def __init__(self, x: float, y: float):
        # TODO: Initialize components as floats
        self.x = float(x)
        self.y = float(y)

    def __repr__(self) -> str:
        # TODO: Return Vector2D(x, y)
        return f"Vector2D({self.x}, {self.y})"

    def __str__(self) -> str:
        # TODO: Return (x, y)
        return f"({self.x}, {self.y})"

    def __add__(self, other):
        # TODO: Implement addition
        if isinstance(other, Vector2D):
            return Vector2D(self.x + other.x, self.y + other.y)
        elif isinstance(other, (int, float)):
            return Vector2D(self.x + other, self.y + other)
        return NotImplemented

    def __radd__(self, other):
        # TODO: Implement right-hand addition
        return self.__add__(other)

    def __sub__(self, other):
        # TODO: Implement subtraction
        if isinstance(other, Vector2D):
            return Vector2D(self.x - other.x, self.y - other.y)
        elif isinstance(other, (int, float)):
            return Vector2D(self.x - other, self.y - other)
        return NotImplemented

    def __rsub__(self, other):
        # TODO: Implement right-hand subtraction (e.g. scalar - self)
        if isinstance(other, (int, float)):
            return Vector2D(other - self.x, other - self.y)
        return NotImplemented

    def __mul__(self, scalar):
        # TODO: Implement multiplication
        if isinstance(scalar, (int, float)):
            return Vector2D(self.x * scalar, self.y * scalar)
        return NotImplemented

    def __rmul__(self, scalar):
        # TODO: Implement right-hand multiplication
        return self.__mul__(scalar)

    def __truediv__(self, scalar):
        # TODO: Implement division
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        if scalar == 0:
            raise ZeroDivisionError("Cannot divide vector by zero")
        return Vector2D(self.x / scalar, self.y / scalar)

    def __abs__(self) -> float:
        # TODO: Return magnitude
        return math.sqrt(self.x**2 + self.y**2)

    def __eq__(self, other) -> bool:
        # TODO: Implement comparison
        if not isinstance(other, Vector2D):
            return False
        return self.x == other.x and self.y == other.y

In [ ]:
# Test Challenge 1
v1 = Vector2D(3, 4)
v2 = Vector2D(1, 2)

# Representation tests
assert repr(v1) == "Vector2D(3.0, 4.0)"
assert str(v1) == "(3.0, 4.0)"

# Addition tests
assert v1 + v2 == Vector2D(4, 6)
assert v1 + 10 == Vector2D(13, 14)
assert 10 + v1 == Vector2D(13, 14)

# Subtraction tests
assert v1 - v2 == Vector2D(2, 2)
assert v1 - 2 == Vector2D(1, 2)
assert 10 - v1 == Vector2D(7, 6)

# Multiplication tests
assert v1 * 2 == Vector2D(6, 8)
assert 3 * v1 == Vector2D(9, 12)

# Division tests
assert v1 / 2 == Vector2D(1.5, 2.0)
try:
    v1 / 0
    assert False, "ZeroDivisionError was not raised!"
except ZeroDivisionError:
    pass

# Absolute magnitude
assert abs(v1) == 5.0

# Equality
assert v1 == Vector2D(3, 4)
assert v1 != v2

print("Challenge 1 passed!")

### Challenge 2: Dynamic Dot-Access Configuration Dictionary

Implement a class `Config` that acts as a nested read-write dictionary wrapper allowing dot-attribute access:
1. Initialize with an underlying dictionary: `Config(data_dict)`.
2. Attribute lookup (`config.key`): Returns value under `key`. If the value is a dictionary, wrap it dynamically inside a `Config` instance.
3. Missing key lookup: Raise an `AttributeError` for attribute dot-access, and `KeyError` for brackets index access.
4. Attribute mutation (`config.key = val`): Modify or add key in the wrapped dictionary.
5. Emulate containers: support `__getitem__` and `__setitem__` to allow bracket accesses (`config['key']`).

In [ ]:
class Config:
    def __init__(self, data: dict):
        # Use super to store configuration data bypass custom setters
        super().__setattr__('_data', data)

    def __getattr__(self, name):
        # TODO: Handle dot-lookup. Wrap inner dicts with Config
        if name in self._data:
            val = self._data[name]
            if isinstance(val, dict):
                return Config(val)
            return val
        raise AttributeError(f"No config parameter named {name!r}")

    def __setattr__(self, name, value):
        # TODO: Handle dot-mutation
        self._data[name] = value

    def __getitem__(self, key):
        # TODO: Handle bracket-lookup. Wrap inner dicts with Config
        if key in self._data:
            val = self._data[key]
            if isinstance(val, dict):
                return Config(val)
            return val
        raise KeyError(f"No config key {key!r}")

    def __setitem__(self, key, value):
        # TODO: Handle bracket-mutation
        self._data[key] = value

    def __repr__(self) -> str:
        return f"Config({self._data!r})"

In [ ]:
# Test Challenge 2
settings = {
    "database": {
        "host": "localhost",
        "port": 5432
    },
    "debug": True
}

config = Config(settings)

# Lookup tests
assert config.debug is True
assert isinstance(config.database, Config)
assert config.database.host == "localhost"
assert config.database.port == 5432
assert config["database"]["port"] == 5432

# Mutation tests
config.database.port = 8000
assert settings["database"]["port"] == 8000  # Verify mutation modified underlying dict reference

config.app_name = "PythonApp"
assert settings["app_name"] == "PythonApp"

# Validation errors
try:
    config.nonexistent
    assert False, "Allowed accessing nonexistent attribute"
except AttributeError:
    pass

try:
    config["nonexistent"]
    assert False, "Allowed accessing nonexistent key"
except KeyError:
    pass

print("Challenge 2 passed!")

### Challenge 3: SmartList Container

Implement a custom `SmartList` container that wraps a list of numbers but overrides operations to apply them **element-wise** (similar to NumPy arrays):
1. Initialization: `SmartList(elements)` wraps a iterable list of numbers.
2. Container protocol: `len(s)` returns the count of elements; `s[index]` supports direct indexing AND slice lookups (slicing must return a new `SmartList`).
3. Addition (`+`): 
   - `s1 + s2`: Adds matching elements together. Raise `ValueError` if the lists have different lengths.
   - `s1 + scalar`: Adds numeric value to each element.
4. Multiplication (`*`): `s1 * scalar` multiplies every element by the scalar.

In [ ]:
class SmartList:
    def __init__(self, elements):
        self.elements = list(elements)

    def __len__(self) -> int:
        # TODO: Implement len()
        return len(self.elements)

    def __getitem__(self, index):
        # TODO: Return SmartList if index is a slice object, else the specific item
        if isinstance(index, slice):
            return SmartList(self.elements[index])
        return self.elements[index]

    def __add__(self, other):
        # TODO: Implement element-wise addition
        if isinstance(other, SmartList):
            if len(self) != len(other):
                raise ValueError("Mismatched SmartList lengths for addition")
            return SmartList([a + b for a, b in zip(self.elements, other.elements)])
        elif isinstance(other, (int, float)):
            return SmartList([a + other for a in self.elements])
        return NotImplemented

    def __mul__(self, scalar):
        # TODO: Implement element-wise scalar multiplication
        if isinstance(scalar, (int, float)):
            return SmartList([a * scalar for a in self.elements])
        return NotImplemented

    def __repr__(self) -> str:
        return f"SmartList({self.elements!r})"

In [ ]:
# Test Challenge 3
s1 = SmartList([1, 2, 3])
s2 = SmartList([10, 20, 30])

# Length and repr
assert len(s1) == 3
assert repr(s1) == "SmartList([1, 2, 3])"

# Slicing and indexing
assert s1[1] == 2
sub = s1[1:]
assert isinstance(sub, SmartList)
assert sub.elements == [2, 3]

# Math operator overrides
assert (s1 + s2).elements == [11, 22, 33]
assert (s1 + 5).elements == [6, 7, 8]
assert (s1 * 10).elements == [10, 20, 30]

# Mismatched addition error
try:
    s1 + SmartList([10, 20])
    assert False, "Allowed mismatched addition"
except ValueError:
    pass

print("Challenge 3 passed!")

[◀ Notebook 09: Classes & OOP](09_classes_and_oop.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 11: Type Hinting ▶](11_type_hinting.ipynb)